# 01A — Bronze from the Azure landing zone

Use this notebook after running the one-time local ingestion script and creating a Lakehouse
shortcut named `us-energy-raw` that points to the deployed Storage account's `raw` container.

The local script writes a validated, Bronze-ready NDJSON batch and updates
`energy/latest.json`. This notebook reads that manifest, validates the expected datasets and
eight-month window, then appends the batch to `bronze.energy_source_raw`.

Do not run both `01_bronze_ingest.ipynb` and this notebook for the same refresh. Choose one
Bronze ingestion path, then continue with notebooks 02 and 03.

In [ ]:
environment = "dev"  # dev | test | prod
raw_shortcut_name = "us-energy-raw"

In [ ]:
from datetime import datetime
import json

import notebookutils
from pyspark.sql import functions as F, types as T

if environment not in {"dev", "test", "prod"}:
    raise ValueError("environment must be dev, test, or prod")
if not raw_shortcut_name or "/" in raw_shortcut_name:
    raise ValueError("raw_shortcut_name must be the Lakehouse shortcut name only.")

spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
manifest_path = f"Files/{raw_shortcut_name}/energy/latest.json"
try:
    manifest = json.loads(notebookutils.fs.head(manifest_path, 1024 * 1024))
except Exception as exc:
    raise RuntimeError(
        f"Unable to read {manifest_path}. Verify the shortcut, Storage RBAC, and that the "
        "one-time local ingestion script completed successfully."
    ) from exc

required_manifest_fields = {
    "batch_id",
    "retrieved_at_utc",
    "selected_periods",
    "row_counts",
    "bronze_records_path",
}
missing_manifest_fields = required_manifest_fields - set(manifest)
if missing_manifest_fields:
    raise RuntimeError(f"Azure landing manifest is missing fields: {sorted(missing_manifest_fields)}")
if len(manifest["selected_periods"]) != 8:
    raise RuntimeError("Azure landing manifest must contain exactly eight selected periods.")

records_path = f"Files/{raw_shortcut_name}/{manifest['bronze_records_path']}"
bronze_df = spark.read.json(records_path)
required_columns = {
    "dataset_name",
    "period",
    "state_code_raw",
    "state_name_raw",
    "state_fips_raw",
    "sector_code_raw",
    "fuel_code_raw",
    "fuel_name_raw",
    "generation_raw",
    "total_consumption_raw",
    "consumption_for_eg_raw",
    "total_consumption_btu_raw",
    "consumption_for_eg_btu_raw",
    "retail_sales_raw",
    "population_raw",
    "generation_units_raw",
    "consumption_for_eg_btu_units_raw",
    "retail_sales_units_raw",
    "source_url",
    "source_units_json",
    "source_metadata_json",
    "batch_id",
    "ingested_at_utc",
    "ingestion_date",
}
missing_columns = required_columns - set(bronze_df.columns)
if missing_columns:
    raise RuntimeError(f"Azure Bronze records are missing columns: {sorted(missing_columns)}")

bronze_df = (
    bronze_df.select(*sorted(required_columns))
    .withColumn("ingested_at_utc", F.to_timestamp("ingested_at_utc"))
    .withColumn("ingestion_date", F.to_date("ingestion_date"))
)
if bronze_df.where(F.col("batch_id") != F.lit(manifest["batch_id"])).limit(1).count():
    raise RuntimeError("Azure Bronze records contain a batch ID different from the latest manifest.")

In [ ]:
counts = {
    row["dataset_name"]: row["count"]
    for row in bronze_df.groupBy("dataset_name").count().collect()
}
required_datasets = {
    "eia_electric_power_operational",
    "eia_retail_sales",
    "census_population_vintage_2021",
}
if set(counts) != required_datasets or any(counts[name] == 0 for name in required_datasets):
    raise RuntimeError(f"Azure Bronze validation failed; source counts: {counts}")

selected_periods = {
    row["period"]
    for row in bronze_df.where(F.col("period").isNotNull()).select("period").distinct().collect()
}
if selected_periods != set(manifest["selected_periods"]):
    raise RuntimeError(
        "Azure Bronze periods do not exactly match the selected periods in the latest manifest."
    )

existing_batch = (
    spark.table("bronze.energy_source_raw")
    .where(F.col("batch_id") == F.lit(manifest["batch_id"]))
    .limit(1)
    .count()
    if spark.catalog.tableExists("bronze.energy_source_raw")
    else 0
)
if existing_batch:
    raise RuntimeError(
        f"Batch {manifest['batch_id']} is already present in bronze.energy_source_raw."
    )

(
    bronze_df.write.format("delta")
    .mode("append")
    .partitionBy("ingestion_date", "dataset_name")
    .saveAsTable("bronze.energy_source_raw")
)
print(
    {
        "batch_id": manifest["batch_id"],
        "counts": counts,
        "selected_periods": manifest["selected_periods"],
        "source": records_path,
    }
)